In [79]:
import torch
import torch.nn as nn
from cauchy import CauchyActivation
import torch.optim as optim
from torchvision import datasets , transforms
from torch.utils.data import DataLoader

In [80]:
activationinstance = CauchyActivation()

In [81]:
class Xnet(nn.Module):
    def __init__(self , activation):
        super().__init__()
        self.activation_function = activation

        self.input_layer = nn.Linear(784 , 128)
        self.hidden_layer = nn.Linear(128 , 128)
        self.output_layer = nn.Linear(128 , 10)

    def forward(self , x):
        x = x.view(-1 , 784)
        x = self.input_layer(x)
        x = self.activation_function(x)
        x = self.hidden_layer(x)
        x = self.activation_function(x)
        x = self.output_layer(x)
        
        return x

In [82]:
func = Xnet(activation = activationinstance)

In [77]:
class Xnet(nn.Module):
    def __init__(self, activation):
        super().__init__()
        # Explicitly register the activation as a submodule
        self.add_module("activation_function", activation)  # <-- Key fix

        self.input_layer = nn.Linear(784, 128)
        self.hidden_layer = nn.Linear(128, 128)
        self.output_layer = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = self.input_layer(x)
        x = self.activation_function(x)  # Now callable
        x = self.hidden_layer(x)
        x = self.activation_function(x)
        x = self.output_layer(x)
        return x

In [84]:
func(torch.rand(1, 28, 28))

TypeError: 'CauchyActivation' object is not callable

In [58]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [60]:
transform = transforms.Compose([transforms.ToTensor()])

train_data = datasets.MNIST(root = './data' , train = True , download = True , transform = transform)
test_data = datasets.MNIST(root = './data' , train = False , download = True , transform = transform)

In [61]:
train_loader = DataLoader(train_data , batch_size = 64 , shuffle = True)
test_loader = DataLoader(test_data , batch_size = 64 , shuffle = True)

In [50]:
x_nets = Xnet(activation = activationinstance).to(device)

In [62]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(x_nets.parameters() , lr = 0.0001)

In [63]:
def train(model , train_loader , optimizer , device):
    model.train()
    total_loss = 0
    correct = 0
    total_samples = 0

    for data, target in train_loader:
        data , target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output , target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = output.argmax(dim = 1 , keepdim = True)
        correct += pred.eq(target.view_as(pred)).sum().item()
        total_samples += target.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total_samples

    return avg_loss , accuracy

In [64]:
def test(model , device , test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    total_samples = 0

    with torch.no_grad():
        for data , target in test_loader:
            data , target = data.to(device) , target.to(device)
            output = model(data)
            batch_loss = criterion(output , target).item()
            test_loss += batch_loss
            pred = output.argmax(dim = 1 , keepdim = True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            total_samples += target.size(0)
        
    test_loss /= len(test_loader)
    accuracy = 100. * correct / total_samples

    return test_loss , accuracy

In [65]:
train_loss, train_accuracy = [] , []
test_loss, test_accuracy = [] , []

epochs = 20

for epoch in range(1, epochs + 1):
    tra_loss, tra_acc = train(x_nets, train_loader , optimizer , device)
    testd_loss, testd_acc = test(x_nets, device, test_loader)
    train_loss.append(tra_loss)
    train_accuracy.append(tra_acc)
    test_loss.append(testd_loss)
    test_accuracy.append(testd_acc)

    print(f'Epoch {epoch:02d} - Training Loss: {train_loss:.6f}, Training Acc: {train_accuracy:.2f}%, Validation Loss: {testd_loss:.6f}, Validation Acc: {testd_acc:.2f}%')

TypeError: 'CauchyActivation' object is not callable